# Operations Manager Assignment: Candidate Data Processing

This notebook processes raw candidate data from an Excel file (`candidates_raw.xlsx`), performing cleaning, deduplication, merging, and eligibility assessment. The final output is a structured CSV file (`task1.csv`) containing processed candidate information.

## Section 1: Imports

In [1]:
import pandas as pd
import re
from dateutil import parser as date_parser
from collections import defaultdict
import numpy as np
import os

## Section 2: Read Excel

In [2]:
try:
    # Read the Excel file
    df_raw = pd.read_excel('candidates_raw.xlsx')

    # Display shape and preview
    print(f"Original DataFrame shape: {df_raw.shape}")
    print("\nPreview of original DataFrame:")
    display(df_raw.head())
except FileNotFoundError:
    print("Error: 'candidates_raw.xlsx' not found. Please upload the file to the Colab environment.")
    df_raw = pd.DataFrame() # Create an empty DataFrame to avoid further errors


Original DataFrame shape: (40, 7)

Preview of original DataFrame:


,Name,Phone,Email,City,Total Experience,Date Applied,Current Company
0,Ishita Agarwal,9840012310,ishita.agarwal@gmail.com,"Hyderabad, Telangana",3.5 yrs,2026-07-11,Accenture Ops
1,Bhavana Reddy,98400 12304,bhavana.reddy@yahoo.com,Hyderabad,2.5 years,2026-07-05,HGS
2,JYOTI PATEL,+91 98400 12311,jyoti.patel@yahoo.com,Hyderabad,1.2,2026-07-12,Firstsource
3,kiran kumar,098400 12334,KIRAN.KUMAR@GMAIL.COM,hyderabad,4 years,2026-12-07,Fresher (2026 graduate)
4,SRINIVAS RAO,+91 98400 12323,srinivas.rao@gmail.com,Hyderabad,3 yrs,2026-07-11,Vee Healthtek


## Section 3: Cleaning Functions

In [3]:
def normalize_phone(phone_str):
    """
    Normalizes a phone number to a bare 10-digit string.
    Removes country codes (+91), leading zeros, spaces, hyphens, brackets, and any other non-digit characters.
    If the normalized number is not 10 digits, returns an empty string.
    """
    if pd.isna(phone_str):
        return ''
    # Convert to string and remove all non-digit characters
    digits = re.sub(r'[^\d]', '', str(phone_str))

    # Remove leading +91 or other country codes, assuming 10-digit standard
    if len(digits) > 10 and digits.startswith('91'):
        digits = digits[2:] # Remove '91'

    # Remove leading zero if present and if it's longer than 10 digits (e.g. 09840012311 -> 9840012311)
    if len(digits) > 10 and digits.startswith('0'):
        digits = digits[1:]

    # Ensure it's exactly 10 digits
    if len(digits) == 10:
        return digits
    return ''

def normalize_email(email_str):
    """
    Normalizes an email address: trims whitespace and converts to lowercase.
    Returns an empty string if missing.
    """
    if pd.isna(email_str):
        return ''
    return str(email_str).strip().lower()

def normalize_name(name_str):
    """
    Normalizes a name: converts to Title Case and removes extra spaces.
    Returns an empty string if missing.
    """
    if pd.isna(name_str):
        return ''
    # Remove extra spaces and convert to Title Case
    return ' '.join(str(name_str).split()).title()

def normalize_city(city_str):
    """
    Normalizes a city name: converts to Title Case and keeps only the city part
    (e.g., 'Hyderabad, Telangana' -> 'Hyderabad').
    Returns an empty string if missing.
    """
    if pd.isna(city_str):
        return ''
    city = str(city_str).split(',')[0].strip()
    return city.title()

def normalize_experience(exp_str):
    """
    Normalizes experience to years, rounded to one decimal place.
    Handles formats like '3 yrs', '3 years', '2', '18 months', '8 months', 'Fresher'.
    Returns 0.0 for 'Fresher' or if parsing fails.
    """
    if pd.isna(exp_str):
        return 0.0
    s = str(exp_str).lower().strip()

    if 'fresher' in s:
        return 0.0

    try:
        # Handle 'years' or 'yrs'
        if 'year' in s or 'yr' in s:
            num_str = re.findall(r'\d+\.?\d*', s)[0]
            return round(float(num_str), 1)
        # Handle 'months'
        elif 'month' in s:
            num_str = re.findall(r'\d+\.?\d*', s)[0]
            return round(float(num_str) / 12, 1)
        # Handle bare numbers
        elif re.fullmatch(r'\d+\.?\d*', s):
            return round(float(s), 1)
    except (IndexError, ValueError):
        pass # Fall through to return 0.0 if parsing fails
    return 0.0

def normalize_date(date_str):
    """
    Normalizes a date string to YYYY-MM-DD format.
    Handles various date formats using dateutil.parser. Safely returns NaT if parsing fails.
    """
    if pd.isna(date_str):
        return pd.NaT
    try:
        # dateutil.parser is robust for various formats
        return date_parser.parse(str(date_str)).strftime('%Y-%m-%d')
    except (ValueError, TypeError):
        return pd.NaT # Return Not a Time for invalid dates


## Section 4: Apply Cleaning

In [4]:
if not df_raw.empty:
    df = df_raw.copy()

    df['full_name'] = df['Name'].apply(normalize_name)
    df['phone'] = df['Phone'].apply(normalize_phone)
    df['email'] = df['Email'].apply(normalize_email)
    df['city'] = df['City'].apply(normalize_city)
    df['experience_years'] = df['Total Experience'].apply(normalize_experience)
    df['date_applied'] = df['Date Applied'].apply(normalize_date)

    # Add a 'remarks' column for later use
    df['remarks'] = ''

    print("\nPreview of DataFrame after cleaning:")
    display(df[['full_name', 'phone', 'email', 'city', 'experience_years', 'date_applied', 'remarks']].head())
else:
    print("DataFrame is empty, skipping cleaning step.")
    df = pd.DataFrame(columns=['full_name', 'phone', 'email', 'city', 'experience_years', 'date_applied', 'remarks'])



Preview of DataFrame after cleaning:


,full_name,phone,email,city,experience_years,date_applied,remarks
0,Ishita Agarwal,9840012310,ishita.agarwal@gmail.com,Hyderabad,3.5,2026-07-11,
1,Bhavana Reddy,9840012304,bhavana.reddy@yahoo.com,Hyderabad,2.5,2026-07-05,
2,Jyoti Patel,9840012311,jyoti.patel@yahoo.com,Hyderabad,1.2,2026-07-12,
3,Kiran Kumar,9840012334,kiran.kumar@gmail.com,Hyderabad,4.0,2026-12-07,
4,Srinivas Rao,9840012323,srinivas.rao@gmail.com,Hyderabad,3.0,2026-07-11,


## Section 5: Union-Find Duplicate Detection

In [5]:
if not df.empty:
    class UnionFind:
        """
        Implements the Union-Find (Disjoint Set Union) data structure.
        Used to group connected components, essential for transitive duplicate detection.
        """
        def __init__(self, n):
            self.parent = list(range(n))
            self.rank = [0] * n # For optimization: union by rank

        def find(self, i):
            """
            Finds the representative (root) of the set containing element `i`.
            Performs path compression for efficiency.
            """
            if self.parent[i] == i:
                return i
            self.parent[i] = self.find(self.parent[i])
            return self.parent[i]

        def union(self, i, j):
            """
            Unites the sets containing elements `i` and `j`.
            Performs union by rank for efficiency.
            Returns True if a union was performed, False otherwise (they were already in the same set).
            """
            root_i = self.find(i)
            root_j = self.find(j)

            if root_i != root_j:
                if self.rank[root_i] < self.rank[root_j]:
                    self.parent[root_i] = root_j
                elif self.rank[root_i] > self.rank[root_j]:
                    self.parent[root_j] = root_i
                else:
                    self.parent[root_j] = root_i
                    self.rank[root_i] += 1
                return True
            return False

    uf = UnionFind(len(df))

    # Map normalized phones/emails to their first observed index
    phone_to_idx = defaultdict(list)
    email_to_idx = defaultdict(list)

    for i, row in df.iterrows():
        if row['phone']:
            phone_to_idx[row['phone']].append(i)
        if row['email']:
            email_to_idx[row['email']].append(i)

    # Union based on phone matches
    for phone, indices in phone_to_idx.items():
        if len(indices) > 1:
            # Union all records sharing the same phone number
            for i in range(1, len(indices)):
                uf.union(indices[0], indices[i])

    # Union based on email matches
    for email, indices in email_to_idx.items():
        if len(indices) > 1:
            # Union all records sharing the same email address
            for i in range(1, len(indices)):
                uf.union(indices[0], indices[i])

    # Group records by their representative (root) in the Union-Find structure
    duplicate_groups = defaultdict(list)
    for i in range(len(df)):
        root = uf.find(i)
        duplicate_groups[root].append(i)

    print(f"Found {len(duplicate_groups)} unique candidate groups after deduplication.")
else:
    print("DataFrame is empty, skipping duplicate detection.")
    duplicate_groups = {}


Found 37 unique candidate groups after deduplication.


## Section 6: Merge Duplicate Groups

In [6]:
if not df.empty and duplicate_groups:
    merged_candidates = []

    for root, indices in duplicate_groups.items():
        group_df = df.iloc[indices]

        # Initialize merged record with values from the first record in the group
        # This is a baseline, individual rules will override
        merged_record = group_df.iloc[0].to_dict()

        # Flag to check if any merge actually happened in this group
        merged_flag = False

        # Merging Rules:
        # full_name: most complete (longest string, or first non-empty if lengths are equal)
        full_names = group_df['full_name'].dropna().tolist()
        if full_names:
            merged_record['full_name'] = max(full_names, key=len) if full_names else ''

        # phone: first non-empty
        phones = group_df['phone'].loc[group_df['phone'] != ''].tolist()
        if phones:
            merged_record['phone'] = phones[0]

        # email: first non-empty
        emails = group_df['email'].loc[group_df['email'] != ''].tolist()
        if emails:
            merged_record['email'] = emails[0]

        # city: most complete (longest string, or first non-empty if lengths are equal)
        cities = group_df['city'].dropna().tolist()
        if cities:
            merged_record['city'] = max(cities, key=len) if cities else ''

        # experience_years: maximum
        merged_record['experience_years'] = group_df['experience_years'].max()

        # date_applied: earliest (convert to datetime for proper comparison)
        dates_applied = pd.to_datetime(group_df['date_applied'], errors='coerce').dropna()
        if not dates_applied.empty:
            merged_record['date_applied'] = dates_applied.min().strftime('%Y-%m-%d')
        else:
            merged_record['date_applied'] = pd.NaT # Reset if no valid date was found

        # Remarks:
        if len(indices) > 1:
            merged_record['remarks'] = 'Merged duplicate records by phone/email.'
            merged_flag = True
        else:
            merged_record['remarks'] = '' # Ensure it's blank if not merged

        merged_candidates.append(merged_record)

    df_deduplicated = pd.DataFrame(merged_candidates)

    # Ensure date_applied column is of type object for string storage 'YYYY-MM-DD'
    # Or convert to datetime and then format if it needs to be datetime objects
    df_deduplicated['date_applied'] = df_deduplicated['date_applied'].apply(lambda x: x if pd.isna(x) else pd.to_datetime(x).strftime('%Y-%m-%d') if isinstance(x, str) else x.strftime('%Y-%m-%d') if pd.notna(x) else pd.NaT)

    print("\nPreview of DataFrame after merging duplicates:")
    display(df_deduplicated[['full_name', 'phone', 'email', 'city', 'experience_years', 'date_applied', 'remarks']].head())
else:
    print("No data to merge or no duplicate groups found.")
    df_deduplicated = pd.DataFrame()



Preview of DataFrame after merging duplicates:


,full_name,phone,email,city,experience_years,date_applied,remarks
0,Ishita Agarwal,9840012310,ishita.agarwal@gmail.com,Hyderabad,3.5,2026-07-11,
1,Bhavana Reddy,9840012304,bhavana.reddy@yahoo.com,Hyderabad,2.5,2026-07-05,
2,Jyoti Patel,9840012311,jyoti.patel@yahoo.com,Hyderabad,1.2,2026-07-12,
3,Kiran Kumar,9840012334,kiran.kumar@gmail.com,Hyderabad,4.0,2026-12-07,
4,Srinivas Rao,9840012323,srinivas.rao@gmail.com,Hyderabad,3.0,2026-07-11,


## Section 7: Eligibility

In [7]:
if not df_deduplicated.empty:
    df_deduplicated['status'] = 'INELIGIBLE'
    df_deduplicated['reason_code'] = ''

    # Apply eligibility rules
    for idx, row in df_deduplicated.iterrows():
        eligible_exp = row['experience_years'] >= 1.0
        eligible_city = row['city'] == 'Hyderabad'

        if eligible_exp and eligible_city:
            df_deduplicated.loc[idx, 'status'] = 'ELIGIBLE'
            df_deduplicated.loc[idx, 'reason_code'] = ''
        else:
            reasons = []
            if not eligible_exp:
                reasons.append('R1') # Experience less than 1 year
            if not eligible_city:
                if row['city'] == 'Pune':
                    reasons.append('R3') # Candidate from Pune
                else:
                    reasons.append('R2') # Outside Hyderabad
            df_deduplicated.loc[idx, 'reason_code'] = ','.join(reasons)

            # Ambiguous data remarks if applicable (e.g. invalid experience or city after cleaning)
            if row['experience_years'] == 0.0 and 'R1' not in reasons and row['Total Experience'] not in ['Fresher', 'fresher']:
                df_deduplicated.loc[idx, 'remarks'] += ' Ambiguous experience data.'
            if not row['city'] and 'R2' not in reasons and row['City']:
                df_deduplicated.loc[idx, 'remarks'] += ' Ambiguous city data.'


    print("\nPreview of DataFrame with eligibility status:")
    display(df_deduplicated[['full_name', 'experience_years', 'city', 'status', 'reason_code', 'remarks']].head())
else:
    print("DataFrame is empty, skipping eligibility check.")



Preview of DataFrame with eligibility status:


,full_name,experience_years,city,status,reason_code,remarks
0,Ishita Agarwal,3.5,Hyderabad,ELIGIBLE,,
1,Bhavana Reddy,2.5,Hyderabad,ELIGIBLE,,
2,Jyoti Patel,1.2,Hyderabad,ELIGIBLE,,
3,Kiran Kumar,4.0,Hyderabad,ELIGIBLE,,
4,Srinivas Rao,3.0,Hyderabad,ELIGIBLE,,


## Section 8: Candidate IDs

In [8]:
if not df_deduplicated.empty:
    # Sort alphabetically by full_name for ID assignment
    df_deduplicated_sorted = df_deduplicated.sort_values(by='full_name').reset_index(drop=True)

    # Assign candidate_id
    df_deduplicated_sorted['candidate_id'] = [f'C{i:03d}' for i in range(1, len(df_deduplicated_sorted) + 1)]

    # Final sort by candidate_id
    df_final = df_deduplicated_sorted.sort_values(by='candidate_id').reset_index(drop=True)

    print("\nPreview of final DataFrame with Candidate IDs:")
    display(df_final[['candidate_id', 'full_name', 'status', 'reason_code']].head())
else:
    print("DataFrame is empty, skipping candidate ID assignment.")
    df_final = pd.DataFrame()



Preview of final DataFrame with Candidate IDs:


,candidate_id,full_name,status,reason_code
0,C001,Aarti Deshmukh,ELIGIBLE,
1,C002,Abhishek Rao,ELIGIBLE,
2,C003,Amit Joshi,INELIGIBLE,R3
3,C004,Anil Kumar,ELIGIBLE,
4,C005,Arjun Nair,INELIGIBLE,R2


## Section 9: Export CSV

In [9]:
if not df_final.empty:
    output_folder = 'output'
    output_filename = os.path.join(output_folder, 'task1.csv')

    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)

    # Define exact output columns
    output_columns = [
        'candidate_id',
        'full_name',
        'phone',
        'email',
        'city',
        'experience_years',
        'date_applied',
        'status',
        'reason_code',
        'remarks'
    ]

    # Ensure all output columns exist, adding them with empty values if not
    for col in output_columns:
        if col not in df_final.columns:
            df_final[col] = ''

    # Export to CSV
    df_final[output_columns].to_csv(output_filename, index=False)

    print(f"Processed data exported to '{output_filename}'")
else:
    print("Final DataFrame is empty, no CSV exported.")


Processed data exported to 'output/task1.csv'


## Section 10: Summary

In [10]:
if not df_raw.empty:
    print("\n--- Processing Summary ---")
    print(f"Original rows: {df_raw.shape[0]}")
    print(f"Rows after deduplication: {df_deduplicated.shape[0]}")

    if not df_final.empty:
        eligible_candidates = df_final[df_final['status'] == 'ELIGIBLE'].shape[0]
        ineligible_candidates = df_final[df_final['status'] == 'INELIGIBLE'].shape[0]
        print(f"Eligible candidates: {eligible_candidates}")
        print(f"Ineligible candidates: {ineligible_candidates}")
    else:
        print("No final candidates to summarize eligibility.")
else:
    print("No original data to summarize.")



--- Processing Summary ---
Original rows: 40
Rows after deduplication: 37
Eligible candidates: 29
Ineligible candidates: 8
